# Lesson 0b: Introduction to Deep Learning — Practical

The companion theory notebook (0a) proved, by hand, that a linear model
cannot solve XOR, and hand-derived the weights of a two-layer network that
can. It deliberately used no autograd and no training — the weights were
picked by inspection to make a point about *representational capacity*.

This notebook takes the same shape — a small stack of linear layers with a
non-linearity between them — and replaces "derive weights by hand" with "let
gradient descent find them", using the production framework (PyTorch)
instead of hand-rolled NumPy. We then point that same machinery at a real
dataset, FashionMNIST, and train a small classifier end to end.

By the end of this notebook you will have:
- used PyTorch tensors and autograd to compute gradients automatically,
- built a network with `nn.Module` instead of hand-written matrix algebra,
- written a generic training-loop skeleton (model / optimizer / loss / loop
  over batches) that later lessons in this series reuse, and
- trained that network on FashionMNIST and reported test accuracy.

## Introduction

Lesson 0a's argument for depth was structural: stack non-linear layers and
you can represent shapes a linear model cannot. But that notebook picked the
weights by inspection — a trick that only works because XOR has four data
points and a designer who already knows the answer.

Real problems don't come with hand-derivable weights. FashionMNIST — grayscale
28x28 photos of clothing, 10 categories — has 60,000 training images. No one
is going to stare at those and write down a weight matrix. Instead we need:

1. a way to compute how each weight should change to reduce error
   (**autograd**), and
2. a way to describe "the right shape of network" cleanly, so we're not
   hand-writing `X @ W1 + b1` for every layer (**`nn.Module`**).

Both of those come from the framework, not from us. That's the whole point
of "the production framework": PyTorch turns the theory notebook's manual
calculus into two lines of code.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, data
# shuffling, minibatch order) is reproducible. Seed both numpy and torch,
# and do it before anything random happens.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import torch.nn as nn
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5, 4)
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("numpy:", np.__version__)

### Device check

This notebook is written to run unmodified in Google Colab or locally. If a
GPU is available (as it typically is on a Colab GPU runtime) we use it;
otherwise we fall back to CPU. Every tensor and module below is moved to
`device` explicitly, so this cell is the only place that decides.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)

## Tensors and Autograd

A `torch.Tensor` is NumPy's `ndarray` with two superpowers relevant here:
it can live on a GPU, and it can remember the graph of operations used to
compute it, so that gradients can be computed automatically.

### From NumPy arrays to tensors

Lesson 0a's forward pass was `z1 = X @ W1 + b1` on NumPy arrays. The same
line works almost unchanged on tensors — the difference is what happens
next.

In [ ]:
# The same XOR data from lesson 0a, as tensors instead of NumPy arrays.
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y = torch.tensor([0., 1., 1., 0.])

W1 = torch.randn(2, 2, requires_grad=True)
b1 = torch.zeros(2, requires_grad=True)

z1 = X @ W1 + b1
print("z1:", z1)
print("z1 requires_grad:", z1.requires_grad)
print("z1 grad_fn:", z1.grad_fn)

`requires_grad=True` tells PyTorch to track every operation performed on
`W1` and `b1`. The result, `z1`, carries a `grad_fn` — a pointer back into
that computational graph (the same graph lesson 0a drew as a picture). We
have not computed any gradient yet; we've only recorded *how* to.

### `.backward()` and `.grad`

Autograd's job is to turn a scalar loss into a gradient for every tensor
that has `requires_grad=True`, by walking the recorded graph backward
(reverse-mode automatic differentiation — the calculus behind every neural
network training loop). Calling `.backward()` on a scalar does exactly that;
the results land in each leaf tensor's `.grad` attribute.

In [ ]:
a1 = torch.sigmoid(z1)
loss = ((a1.sum(dim=1) - y) ** 2).mean()
print("loss:", loss.item())

loss.backward()
print("dLoss/dW1:")
print(W1.grad)
print("dLoss/db1:")
print(b1.grad)

No calculus was written by hand — `.backward()` computed
$\partial \text{loss} / \partial W_1$ and $\partial \text{loss} /
\partial b_1$ by applying the chain rule through every operation between
`W1` and `loss`. This is the machinery that makes training networks with
millions of parameters tractable: the same `.backward()` call scales from
this four-point toy example to FashionMNIST below.

One detail matters in practice: gradients **accumulate** in `.grad` by
default, so a real training loop must zero them before each new
`.backward()` call (the training loop below does this via
`optimizer.zero_grad()`).

### `nn.Module`: describing networks without writing matrix algebra

Lesson 0a wrote `W1`, `b1`, `W2`, `b2` and the forward pass by hand.
`nn.Module` packages a layer's parameters and its forward computation
together, and `nn.Sequential` chains layers into a network — the same
"stack of linear layers with non-linearities between them" shape from 0a,
now written declaratively.

In [ ]:
tiny_net = nn.Sequential(
    nn.Linear(2, 2),
    nn.Sigmoid(),
    nn.Linear(2, 1),
    nn.Sigmoid(),
)

out = tiny_net(X)
print("output shape:", out.shape)
print("parameters:")
for name, param in tiny_net.named_parameters():
    print(f"  {name}: {tuple(param.shape)}")

Every `nn.Linear` already carries `requires_grad=True` weights and
biases, initialised randomly (seeded above) rather than picked by hand.
`tiny_net(X)` runs the forward pass; `.parameters()` exposes everything an
optimizer needs to update. This is the same two-layer shape as lesson 0a's
`forward()` function, but the framework now owns the bookkeeping.

## The Training Loop

Every network trained in this curriculum — from this tiny two-layer net to
the transformers built much later — follows the same four-part loop:

1. **model** — an `nn.Module` producing predictions from inputs
2. **loss** — a function comparing predictions to targets
3. **optimizer** — a rule for turning gradients into parameter updates
4. **loop over batches** — repeat (forward → loss → backward → step) over
   the data, for a number of epochs

We write it once, generically, below. Later lessons reuse this exact
skeleton with a different `model` and `loss_fn`.

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    """One pass over `loader`, updating `model`'s parameters. Returns the
    mean training loss for the epoch."""
    model.train()
    total_loss, n_batches = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()          # gradients accumulate — clear first
        preds = model(xb)              # forward pass
        loss = loss_fn(preds, yb)      # compare to targets
        loss.backward()                # autograd: fill in .grad for every param
        optimizer.step()               # apply the update rule

        total_loss += loss.item()
        n_batches += 1
    return total_loss / n_batches


@torch.no_grad()
def evaluate(model, loader, device):
    """Mean accuracy of `model` over `loader`, with no gradient tracking."""
    model.eval()
    correct, total = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb)
        correct += (preds.argmax(dim=1) == yb).sum().item()
        total += yb.size(0)
    return correct / total

`@torch.no_grad()` turns off autograd's bookkeeping for evaluation —
we don't need gradients when we're only measuring accuracy, and skipping
graph construction is both faster and lighter on memory. This
train/evaluate pair is the generic skeleton; everything below plugs a
concrete model, optimizer, and dataset into it.

## Training on FashionMNIST

FashionMNIST replaces MNIST's digits with 28x28 grayscale photos of
clothing across 10 categories (t-shirt, trouser, pullover, ...) — harder
than digit recognition, still small enough to train in minutes on a CPU.

`torchvision.datasets.FashionMNIST` downloads the dataset automatically the
first time this cell runs, to a local `data/` folder next to this notebook
(this works identically in Colab and locally — no manual setup).

To keep the whole notebook well under the runtime budget on a CPU, we
subsample heavily: a few thousand training images and a thousand test
images, a couple of epochs, and a small network. This is a teaching
example, not a leaderboard run — the point is to see the training-loop
skeleton above actually reduce loss and produce a working classifier, not
to squeeze out the last percent of accuracy.

In [ ]:
transform = transforms.Compose([transforms.ToTensor()])

train_full = torchvision.datasets.FashionMNIST(
    root="data", train=True, download=True, transform=transform
)
test_full = torchvision.datasets.FashionMNIST(
    root="data", train=False, download=True, transform=transform
)

class_names = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# Subsample heavily: this keeps the whole notebook well under the 10-minute
# CPU budget while still demonstrating real training on real data.
N_TRAIN, N_TEST = 4000, 1000
generator = torch.Generator().manual_seed(SEED)
train_indices = torch.randperm(len(train_full), generator=generator)[:N_TRAIN]
test_indices = torch.randperm(len(test_full), generator=generator)[:N_TEST]

train_ds = Subset(train_full, train_indices.tolist())
test_ds = Subset(test_full, test_indices.tolist())

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, generator=generator)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

print(f"train subset: {len(train_ds)} images, test subset: {len(test_ds)} images")

In [ ]:
# A look at the data before we train on it.
fig, axes = plt.subplots(1, 6, figsize=(12, 2.2))
for ax, idx in zip(axes, train_indices[:6].tolist()):
    img, label = train_full[idx]
    ax.imshow(img.squeeze(0), cmap="gray")
    ax.set_title(class_names[label], fontsize=9)
    ax.axis("off")
plt.suptitle("FashionMNIST samples")
plt.show()

### The model

A small multi-layer perceptron (MLP): flatten the 28x28 image to a 784-long
vector, then two hidden layers with ReLU activations, then a 10-way output
(one logit per clothing category). This is exactly the "stack of linear
layers with non-linearities between them" shape from lesson 0a and the
`tiny_net` above — deeper, and wider, but the same idea — sized small (two
hidden layers, width capped at 128) to stay fast on a CPU.

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features=28 * 28, hidden=128, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"total parameters: {n_params:,}")

### Training

We reuse `train_one_epoch` and `evaluate` from above unchanged — this is
the payoff of writing the loop generically. A couple of epochs over the
subsampled training set is enough to see the loss drop and the model
clearly outperform random guessing (10% for 10 balanced classes).

In [ ]:
N_EPOCHS = 3
train_losses = []

for epoch in range(1, N_EPOCHS + 1):
    epoch_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    train_losses.append(epoch_loss)
    print(f"epoch {epoch}/{N_EPOCHS} — mean training loss: {epoch_loss:.4f}")

test_accuracy = evaluate(model, test_loader, device)
print(f"\ntest accuracy on {len(test_ds)} held-out images: {test_accuracy:.1%}")

In [ ]:
plt.plot(range(1, N_EPOCHS + 1), train_losses, marker="o")
plt.xlabel("epoch")
plt.ylabel("mean training loss")
plt.title("Training loss (FashionMNIST MLP)")
plt.xticks(range(1, N_EPOCHS + 1))
plt.grid(alpha=0.3)
plt.show()

### A look at the predictions

Numbers on their own hide *what kind* of mistakes the model makes. A grid
of sample test images with predicted vs. true label makes that concrete —
correct predictions in black, mistakes in red.

In [ ]:
model.eval()
sample_indices = list(range(8))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))

with torch.no_grad():
    for ax, i in zip(axes.ravel(), sample_indices):
        img, true_label = test_ds[i]
        pred_label = model(img.unsqueeze(0).to(device)).argmax(dim=1).item()
        ax.imshow(img.squeeze(0), cmap="gray")
        color = "black" if pred_label == true_label else "crimson"
        ax.set_title(
            f"pred: {class_names[pred_label]}\ntrue: {class_names[true_label]}",
            fontsize=8, color=color,
        )
        ax.axis("off")

plt.suptitle("Sample predictions (red = wrong)")
plt.tight_layout()
plt.show()

## Key Takeaways

- **Autograd replaces hand-derived calculus.** Lesson 0a hand-derived
  weights for a four-point toy problem; `.backward()` computes exact
  gradients for a network of any size by walking the recorded
  computational graph — the same graph 0a drew as a picture, now built and
  differentiated automatically.
- **`nn.Module` replaces hand-written matrix algebra.** `nn.Linear` and
  `nn.Sequential` describe "a stack of linear layers with non-linearities
  between them" declaratively; the framework owns parameter storage,
  initialisation, and the forward pass.
- **The training loop is a reusable skeleton**: model, loss, optimizer,
  and a loop over batches calling zero_grad → forward → backward → step.
  `train_one_epoch` and `evaluate` above are written generically and will
  be reused, unchanged in shape, by later lessons in this series.
- **The same shape, trained instead of derived, works on real data.** A
  small MLP — two hidden layers, comparable in spirit to 0a's two-layer
  network — reached the test accuracy printed above on FashionMNIST after
  `N_EPOCHS` epochs over an `N_TRAIN`-image training subsample, well above
  the 10% a random guesser would get across 10 balanced classes.
- The device-check pattern (`torch.device("cuda" if torch.cuda.is_available()
  else "cpu")`) used throughout this notebook is the standard way to write
  code that runs unmodified on a Colab GPU runtime or a local CPU.